# Cognitive Inequality Research — Full Pipeline (MIDUS Refresher 2)

Primary dataset: MIDUS Refresher 2 (MR2) — BTACT cognitive battery + validated psychosocial scales.

Pipeline stages:
1. Load MIDUS MR2 (Survey + Cognitive SAV files)
2. PC algorithm causal discovery (+ optional Ollama LLM validation)
3. Baron-Kenny mediation with 1000-iteration bootstrap CIs
4. XGBoost + SHAP prediction model
5. Counterfactual intervention simulation (+1 SD on significant mediators)
6. E-value sensitivity analysis

**Key finding**: Purpose in Life, Sense of Control, and Neighborhood Quality
are significant mediators of the SES→cognition gap (all bootstrap 95% CIs exclude zero).
Together they explain ~37% of the total SES effect on cognitive scores.

In [ ]:
# Cell 1: Install dependencies
!pip install -q causal-learn xgboost shap statsmodels pyreadstat pandas numpy scikit-learn requests networkx pydantic matplotlib

In [ ]:
# Cell 2: Mount Google Drive and set paths
import sys
from pathlib import Path

# --- Google Colab: uncomment these 3 lines ---
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = Path('/content/drive/MyDrive/cognitive-inequality-research')

# --- Local use ---
PROJECT_ROOT = Path('.')
sys.path.insert(0, str(PROJECT_ROOT))

SURVEY_PATH = str(PROJECT_ROOT / 'MR2_P1_SURVEY_N2154_20251003.sav')
COG_PATH    = str(PROJECT_ROOT / 'MR2_P3_BTACT+MOCA_N1934_20260224.sav')
OUTPUT_PATH = str(PROJECT_ROOT / 'data/processed/midus_mr2_cognitive.csv')

print(f'Project root: {PROJECT_ROOT.resolve()}')
print(f'Survey SAV exists:   {Path(SURVEY_PATH).exists()}')
print(f'Cognitive SAV exists: {Path(COG_PATH).exists()}')

In [ ]:
# Cell 3: Load and preprocess MIDUS MR2
from src.data.data_loader_midus_mr2 import load_midus_mr2

df = load_midus_mr2(
    survey_path=SURVEY_PATH,
    cognitive_path=COG_PATH,
    min_age=34, max_age=55,
    output_path=OUTPUT_PATH,
)

MEDIATORS = [
    'depression_score', 'screen_time_change', 'sleep_change', 'has_insurance',
    'purpose_in_life', 'sense_of_control', 'neighborhood_quality',
]
mediators = [m for m in MEDIATORS if m in df.columns]

print(f'N={len(df)}, age range: {df["RB1PRAGE"].min():.0f}–{df["RB1PRAGE"].max():.0f}')
print(f'Mediators: {mediators}')
print(f'Complete cases (all vars): {df[["cognitive_score","ses_index"]+mediators].dropna().shape[0]}')
print()
print(df[['cognitive_score','ses_index']+mediators].describe().round(3))

In [ ]:
# Cell 4: PC Algorithm Causal Discovery
# Set USE_LLM=True if Ollama is running locally (run: ollama serve)
from src.analysis.pc_algorithm import discover_ses_cognition_paths
from src.llm.ollama_client import OllamaClient

USE_LLM = OllamaClient().is_available()
print(f'Ollama available: {USE_LLM}')

pc_result = discover_ses_cognition_paths(
    df, dataset_name='midus_mr2',
    mediators=mediators, alpha=0.05, use_llm=USE_LLM,
)

print('\nAdjacency matrix:')
print(pc_result['adjacency'])
print(f'LLM refined: {pc_result["llm_refined"]}')

if pc_result.get('llm_report'):
    report = pc_result['llm_report']
    print(f'\nLLM validation summary: {report["summary"]}')
    print('Validated edges:')
    for edge in report.get('edges', []):
        print(f'  {edge["source"]} → {edge["target"]}  '
              f'confidence={edge["confidence"]:.2f}  {edge["rationale"]}')

In [ ]:
# Cell 5: Baron-Kenny Mediation with Bootstrap CIs
import numpy as np
from src.analysis.mediation_analysis import (
    analyze_all_mediators, baron_kenny_mediation, get_significant_mediators
)

N_BOOTSTRAP = 1000
weights = df['RB1PWGHT6'].values if 'RB1PWGHT6' in df.columns else None

print('Baron-Kenny path estimates (total effect c=1.42):')
bk_results = {}
for m in mediators:
    bk = baron_kenny_mediation(df, x='ses_index', m=m, y='cognitive_score', weights=weights)
    bk_results[m] = bk
    print(f'  [{m}]  indirect={bk.indirect:.3f}  prop_mediated={bk.proportion_mediated:.3f}')

print(f'\nBootstrap CIs ({N_BOOTSTRAP} iterations):')
boot_results = analyze_all_mediators(
    df, x='ses_index', y='cognitive_score',
    mediators=mediators, weights=weights, n_boot=N_BOOTSTRAP,
)
significant = get_significant_mediators(boot_results)

for m, res in boot_results.items():
    sig = ' *SIGNIFICANT*' if m in significant else ''
    print(f'  {m}: {res.point_estimate:.4f} [{res.ci_lower:.4f}, {res.ci_upper:.4f}]{sig}')

print(f'\nSignificant mediators: {significant}')

In [ ]:
# Cell 6: XGBoost + SHAP
import shap
from src.analysis.prediction_model import CognitivePredictionModel

feature_cols = ['ses_index'] + mediators
model_df = df[feature_cols + ['cognitive_score']].dropna()
X = model_df[feature_cols]
y = model_df['cognitive_score']

model = CognitivePredictionModel()
model.train(X, y)
cv = model.cross_validate(X, y, cv=5)

print(f'CV R²:   {np.mean(cv["r2_scores"]):.3f} ± {np.std(cv["r2_scores"]):.3f}')
print(f'CV RMSE: {np.mean(cv["rmse_scores"]):.3f} ± {np.std(cv["rmse_scores"]):.3f}')

shap_vals = model.compute_shap_values(X)
importance = model.feature_importance()
print('\nFeature importance (mean |SHAP|):')
for feat, imp in importance.items():
    sig_marker = ' *' if feat in significant else ''
    print(f'  {feat}: {imp:.4f}{sig_marker}')

shap.summary_plot(shap_vals, X, show=True)

In [ ]:
# Cell 7: Counterfactual Simulations (+1 SD on significant mediators)
from src.simulation.counterfactual_simulator import generate_interventions

interventions = generate_interventions(boot_results, model, X)

print('Counterfactual interventions (+1 SD shift on significant mediators):')
if not interventions:
    print('  No significant mediators found.')
else:
    for r in interventions:
        pct_change = (r.effect_size / r.baseline_mean) * 100 if r.baseline_mean != 0 else 0
        print(f'  {r.variable}:')
        print(f'    Baseline:       {r.baseline_mean:.3f}')
        print(f'    Counterfactual: {r.counterfactual_mean:.3f}')
        print(f'    Effect:         {r.effect_size:+.3f} ({pct_change:+.1f}%)')

In [ ]:
# Cell 8: E-Value Sensitivity Analysis
from src.analysis.sensitivity_analysis import run_ses_cognition_sensitivity

sd_outcome = float(df['cognitive_score'].std())
sensitivity = run_ses_cognition_sensitivity(bk_results, sd_outcome=sd_outcome)

te = sensitivity['total_effect']
print(f'Total SES effect E-value: {te["evalue_point"]:.2f}  (CI: {te["evalue_ci"]:.2f})')
print(te['interpretation'])
print('\nDirect effect E-values per mediator:')
for med, ev in sensitivity['direct_effects'].items():
    print(f'  {med}: E={ev["evalue_point"]:.2f}  CI E={ev["evalue_ci"]:.2f}')

In [ ]:
# Cell 9: Summary Visualizations
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Cognitive Inequality Research — MIDUS Refresher 2', fontsize=13, fontweight='bold')

# --- SHAP feature importance ---
ax = axes[0]
feats  = list(importance.keys())
vals   = list(importance.values())
colors = ['#1565C0' if f in significant else '#90CAF9' for f in feats]
ax.barh(feats[::-1], vals[::-1], color=colors[::-1])
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Feature Importance (SHAP)')
sig_patch   = mpatches.Patch(color='#1565C0', label='Significant mediator')
insig_patch = mpatches.Patch(color='#90CAF9', label='Non-significant')
ax.legend(handles=[sig_patch, insig_patch], fontsize=8)

# --- Mediation forest plot ---
ax = axes[1]
med_names = list(boot_results.keys())
points = [boot_results[m].point_estimate for m in med_names]
ci_lo  = [boot_results[m].ci_lower for m in med_names]
ci_hi  = [boot_results[m].ci_upper for m in med_names]
y_pos  = list(range(len(med_names)))
for i, (p, lo, hi, m) in enumerate(zip(points, ci_lo, ci_hi, med_names)):
    color = '#C62828' if m in significant else '#BDBDBD'
    ax.plot([lo, hi], [i, i], color=color, linewidth=2.5)
    ax.scatter([p], [i], color=color, zorder=3, s=50)
ax.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(med_names, fontsize=8)
ax.set_xlabel('Indirect effect (95% bootstrap CI)')
ax.set_title('Mediation Forest Plot\n(red = significant, CI excludes 0)')

# --- Counterfactual effects ---
ax = axes[2]
if interventions:
    names   = [r.variable for r in interventions]
    effects = [r.effect_size for r in interventions]
    ax.barh(names[::-1], effects[::-1], color='#2E7D32')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Δ cognitive score (+1 SD intervention)')
    ax.set_title('Counterfactual Interventions\n(significant mediators only)')
else:
    ax.text(0.5, 0.5, 'No significant\ninterventions', ha='center', va='center', fontsize=11)
    ax.set_title('Counterfactual Interventions')

plt.tight_layout()
plt.savefig('pipeline_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved pipeline_summary.png')